# 🐍 Python — Simple to Advanced Examples

A comprehensive tour of Python from first principles to expert patterns.

**Sections:**
1. Built-in types and comprehensions
2. Functions — closures, decorators, generators
3. Classes and the data model
4. Context managers and protocols
5. Concurrency — threading, multiprocessing, asyncio
6. Itertools and functools
7. Type hints and dataclasses
8. Metaclasses and descriptors
9. Memory and performance profiling
10. Design patterns in Python

In [ ]:
# ── 1. Built-in Types and Comprehensions ─────────────────────────────────────

# List comprehension: [expr for item in iterable if condition]
squares     = [x**2 for x in range(10)]              # simple transform
even_squares= [x**2 for x in range(10) if x % 2==0]  # with filter
matrix      = [[i*j for j in range(1,4)] for i in range(1,4)]  # nested

# Dict comprehension: {k: v for ...}
word_lengths = {word: len(word) for word in ['apple','banana','cherry']}

# Set comprehension: unique values
unique_lengths = {len(w) for w in ['cat','dog','bird','fish','ant']}

# Generator expression: lazy evaluation — doesn't build the full list in memory
# Sum of squares of first 1 million numbers — uses O(1) memory
total = sum(x**2 for x in range(1_000_000))

print(f'even squares: {even_squares}')
print(f'matrix: {matrix}')
print(f'word lengths: {word_lengths}')
print(f'sum of squares (1M): {total:,}')

# Walrus operator := — assign and test in one expression (Python 3.8+)
data = [1, 5, 2, 8, 3, 9, 4]
# Filter and keep only values where square > 20, storing the square
big_squares = [sq for x in data if (sq := x**2) > 20]
print(f'squares > 20: {big_squares}')

In [ ]:
# ── 2. Functions — Closures, Decorators, Generators ──────────────────────────
import time, functools

# Closure: inner function remembers variables from the enclosing scope
def make_counter(start=0):
    """Returns a counter function that remembers its state via closure."""
    count = [start]   # mutable container to allow mutation in inner scope
    def counter():
        count[0] += 1
        return count[0]
    return counter

c = make_counter(10)
print(c(), c(), c())   # 11, 12, 13

# Decorator: a function that wraps another function to add behaviour
def timer(func):
    """Decorator that prints the execution time of any function."""
    @functools.wraps(func)   # preserve the original function's metadata
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        print(f'{func.__name__} took {elapsed*1000:.3f}ms')
        return result
    return wrapper

@timer
def slow_sum(n):
    """Sum numbers 0..n-1."""
    return sum(range(n))

result = slow_sum(10_000_000)
print(f'Sum: {result:,}')

# Generator: yields values lazily — infinite sequences without infinite memory
def fibonacci():
    """Infinite Fibonacci sequence generator."""
    a, b = 0, 1
    while True:
        yield a             # suspend here, return a, resume on next()
        a, b = b, a + b

fib = fibonacci()
first_10 = [next(fib) for _ in range(10)]
print(f'Fibonacci: {first_10}')

# send() — coroutine: send values into a generator
def running_average():
    """Coroutine that maintains a running average."""
    total = count = 0
    while True:
        x = yield total / count if count > 0 else 0
        if x is None: break
        total += x; count += 1

avg = running_average()
next(avg)   # prime the coroutine
for val in [10, 20, 30, 40]:
    print(f'After {val}: running avg = {avg.send(val):.1f}')

In [ ]:
# ── 3. Classes and the Python Data Model ─────────────────────────────────────
from __future__ import annotations
import math

class Vector:
    """
    A mathematical vector with overloaded operators.
    Demonstrates __dunder__ (magic) methods that hook into Python's syntax.
    """

    def __init__(self, *components):
        """Accept any number of components: Vector(1, 2, 3)."""
        self.components = tuple(components)

    def __repr__(self):
        """Developer-facing string: repr(v) or just v in REPL."""
        return f'Vector({', '.join(map(str, self.components))})'

    def __len__(self):      return len(self.components)  # len(v)
    def __getitem__(self, i): return self.components[i]  # v[i]

    def __add__(self, other):   # v1 + v2
        return Vector(*(a+b for a,b in zip(self.components, other.components)))

    def __mul__(self, scalar):  # v * 3
        return Vector(*(c * scalar for c in self.components))

    def __rmul__(self, scalar): return self.__mul__(scalar)  # 3 * v

    def __abs__(self):          # abs(v) → Euclidean length
        return math.sqrt(sum(c**2 for c in self.components))

    def dot(self, other):
        """Dot product: Σ aᵢbᵢ"""
        return sum(a*b for a,b in zip(self.components, other.components))

    def normalise(self):
        """Return unit vector."""
        mag = abs(self)
        return Vector(*(c/mag for c in self.components))

v1 = Vector(1, 2, 3)
v2 = Vector(4, 5, 6)
print(f'v1 + v2 = {v1 + v2}')
print(f'v1 * 2  = {v1 * 2}')
print(f'|v1|    = {abs(v1):.4f}')
print(f'v1·v2   = {v1.dot(v2)}')
print(f'v1 norm = {v1.normalise()}')

In [ ]:
# ── 4. Context Managers ───────────────────────────────────────────────────────
from contextlib import contextmanager
import io, time

# Class-based context manager
class Timer:
    """Context manager that measures execution time of a block."""
    def __enter__(self):
        self.start = time.perf_counter()   # called on 'with Timer() as t:'
        return self
    def __exit__(self, *exc_info):         # called on block exit (even if exception)
        self.elapsed = time.perf_counter() - self.start
        return False   # don't suppress exceptions

with Timer() as t:
    total = sum(range(1_000_000))
print(f'sum(range(1M)) = {total:,} in {t.elapsed*1000:.2f}ms')

# Generator-based context manager using @contextmanager
@contextmanager
def managed_resource(name):
    """
    Simulates resource acquisition and release.
    yield marks the boundary between __enter__ and __exit__.
    """
    print(f'→ Acquiring {name}')
    try:
        yield name.upper()   # value bound to 'as' target
    finally:
        print(f'← Releasing {name}')   # always runs, even if exception

with managed_resource('database_connection') as res:
    print(f'  Using {res}')

In [ ]:
# ── 5. Concurrency ────────────────────────────────────────────────────────────
import asyncio, concurrent.futures, threading

# ── asyncio: concurrent I/O without threads ───────────────────────────────────
async def fetch_url(url, delay):
    """Simulate an HTTP fetch that takes 'delay' seconds."""
    await asyncio.sleep(delay)   # non-blocking sleep
    return f'Response from {url} (took {delay}s)'

async def fetch_all():
    """Fetch 3 URLs concurrently — total time ≈ max(delays), not sum."""
    tasks = [
        asyncio.create_task(fetch_url('https://api-a.example', 0.3)),
        asyncio.create_task(fetch_url('https://api-b.example', 0.1)),
        asyncio.create_task(fetch_url('https://api-c.example', 0.2)),
    ]
    results = await asyncio.gather(*tasks)   # run all tasks concurrently
    return results

# Run the event loop
t0 = time.perf_counter()
results = asyncio.run(fetch_all())
print(f'All fetched in {time.perf_counter()-t0:.2f}s (sequential would take 0.6s)')
for r in results: print(f'  {r}')

# ── ThreadPoolExecutor: CPU-bound tasks in threads ────────────────────────────
def cpu_task(n):
    """Simulate CPU work: compute sum of range(n)."""
    return sum(range(n))

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    # Submit 4 tasks, collect futures
    futures = [pool.submit(cpu_task, 10**6) for _ in range(4)]
    thread_results = [f.result() for f in concurrent.futures.as_completed(futures)]
print(f'Thread results (all same): {set(thread_results)}')

In [ ]:
# ── 6. Itertools and Functools ────────────────────────────────────────────────
import itertools, functools

# itertools.chain: iterate over multiple iterables as one
combined = list(itertools.chain([1,2,3], [4,5], [6]))
print(f'chain: {combined}')

# itertools.groupby: group consecutive elements by key
data = [('fruit','apple'),('fruit','banana'),('veg','carrot'),('veg','onion')]
for category, items in itertools.groupby(data, key=lambda x: x[0]):
    print(f'{category}: {[i[1] for i in items]}')

# itertools.product: Cartesian product (nested loops as one expression)
grid = list(itertools.product(['R','G','B'], [1, 2]))
print(f'product: {grid}')

# functools.lru_cache: memoisation decorator
@functools.lru_cache(maxsize=None)   # cache all results
def fib_memo(n):
    """Fibonacci with automatic memoisation — O(n) instead of O(2^n)."""
    if n < 2: return n
    return fib_memo(n-1) + fib_memo(n-2)

print(f'fib(50) = {fib_memo(50):,}')   # instant, no recursion stack overflow

# functools.partial: pre-fill some arguments of a function
def power(base, exponent):
    """Computes base ** exponent."""
    return base ** exponent

square = functools.partial(power, exponent=2)  # pre-fill exponent=2
cube   = functools.partial(power, exponent=3)
print(f'square(5) = {square(5)}, cube(3) = {cube(3)}')

In [ ]:
# ── 7. Type Hints and Dataclasses ─────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import Optional, List, Dict, ClassVar

@dataclass(order=True, frozen=False)
class Employee:
    """
    @dataclass auto-generates __init__, __repr__, __eq__, __lt__ (order=True).
    field() customises individual fields.
    """
    name:       str
    department: str
    salary:     float
    skills:     List[str]         = field(default_factory=list)  # mutable default
    manager:    Optional[str]     = None

    # ClassVar: shared across all instances, not part of __init__
    company: ClassVar[str] = 'Acme Corp'

    def annual_bonus(self, rate: float = 0.1) -> float:
        """Compute bonus as a fraction of salary."""
        return self.salary * rate

    def __post_init__(self):
        """Validation after auto-generated __init__ runs."""
        if self.salary < 0:
            raise ValueError(f'Salary cannot be negative: {self.salary}')

alice = Employee('Alice', 'Engineering', 120_000, ['Python', 'ML'])
bob   = Employee('Bob',   'Engineering',  95_000, ['Java', 'DevOps'], manager='Alice')

print(alice)                           # __repr__ auto-generated
print(f'Alice bonus: ${alice.annual_bonus():,.0f}')
print(f'Alice > Bob (by salary field sort): {alice > bob}')   # order=True

In [ ]:
# ── 8. Metaclasses and Descriptors ────────────────────────────────────────────

# Descriptor: an object that customises attribute access on a class
class Validated:
    """
    Data descriptor that validates a numeric attribute is within [min, max].
    __set_name__ is called when the class is created, binding the attribute name.
    """
    def __set_name__(self, owner, name):
        self.name = name            # e.g. 'age'
        self.private = '_' + name  # e.g. '_age'

    def __init__(self, min_val, max_val):
        self.min_val = min_val
        self.max_val = max_val

    def __get__(self, obj, objtype=None):
        if obj is None: return self   # access via the class itself
        return getattr(obj, self.private, None)

    def __set__(self, obj, value):
        if not (self.min_val <= value <= self.max_val):
            raise ValueError(f'{self.name} must be in [{self.min_val}, {self.max_val}], got {value}')
        setattr(obj, self.private, value)

class Person:
    """Person with validated age and height descriptors."""
    age    = Validated(0, 150)   # descriptor instance assigned to class attribute
    height = Validated(50, 300)  # cm

    def __init__(self, name, age, height):
        self.name   = name
        self.age    = age      # triggers Validated.__set__
        self.height = height

p = Person('Alice', 30, 165)
print(f'{p.name}: age={p.age}, height={p.height}cm')
try:
    p.age = 200   # should raise ValueError
except ValueError as e:
    print(f'Caught: {e}')

In [ ]:
# ── 9. Memory and Performance Profiling ──────────────────────────────────────
import sys, tracemalloc, cProfile, io, pstats

# Memory size of Python objects
print('Object sizes (sys.getsizeof):')
for obj in [42, 3.14, 'hello', [1,2,3], {'a':1}, True]:
    print(f'  {repr(obj):20s} → {sys.getsizeof(obj)} bytes')

# tracemalloc: track exactly where memory is allocated
tracemalloc.start()

# Code under memory analysis
big_list  = [i**2 for i in range(100_000)]   # list: stores all values
big_gen   = (i**2 for i in range(100_000))   # generator: no storage

current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'\nPeak memory: {peak/1024:.1f} KB')
print(f'big_list size: {sys.getsizeof(big_list)/1024:.1f} KB')
print(f'big_gen  size: {sys.getsizeof(big_gen)} bytes  ← generator is tiny')

# cProfile: CPU profiling
def workload():
    """Function to profile."""
    return [x**2 + x**3 for x in range(50_000)]

pr = cProfile.Profile()
pr.enable()
workload()
pr.disable()

stream = io.StringIO()
pstats.Stats(pr, stream=stream).sort_stats('cumulative').print_stats(5)
print('\nTop 5 calls by cumulative time:')
# Print only the stats lines (skip header lines)
for line in stream.getvalue().split('\n')[5:10]: print(line)

In [ ]:
# ── 10. Design Patterns in Python ─────────────────────────────────────────────

# Singleton — ensure only one instance exists
class Singleton:
    """Thread-safe singleton using __new__."""
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

a, b = Singleton(), Singleton()
print(f'Singleton: a is b → {a is b}')   # True

# Observer — publish-subscribe event system
class EventEmitter:
    """Simple pub-sub: register handlers, emit events."""
    def __init__(self):
        self._handlers = {}     # event_name → list of handlers

    def on(self, event, handler):
        self._handlers.setdefault(event, []).append(handler)

    def emit(self, event, *args, **kwargs):
        for h in self._handlers.get(event, []):
            h(*args, **kwargs)

emitter = EventEmitter()
emitter.on('data', lambda x: print(f'  Handler A received: {x}'))
emitter.on('data', lambda x: print(f'  Handler B received: {x*2}'))
print('Emitting data event:')
emitter.emit('data', 42)

# Strategy — swap algorithms at runtime
class Sorter:
    """Sort with a pluggable strategy."""
    def __init__(self, strategy):
        self.strategy = strategy   # callable: (list) → sorted list

    def sort(self, data):
        return self.strategy(data)

data = [5, 2, 8, 1, 9, 3]
asc  = Sorter(sorted)
desc = Sorter(lambda d: sorted(d, reverse=True))
print(f'Ascending:  {asc.sort(data)}')
print(f'Descending: {desc.sort(data)}')